# 3 Aplicação de modelos de Machine Learning para prever o preço médio dos carros

Nesta etapa serão aplicados modelos de regressão para prever o preço médio dos carros (avg_price_brl).  
Serão utilizados os algoritmos Random Forest Regressor e XGBoost Regressor.

As variáveis categóricas serão transformadas em variáveis numéricas para permitir o treinamento dos modelos.

## Preparação dos dados

Antes da aplicação dos modelos de machine learning, é necessário carregar e preparar a base de dados.

Nesta etapa são realizadas as seguintes operações:

- Carregamento da base `precos_carros_brasil.csv`
- Remoção de linhas com valores faltantes
- Remoção de registros duplicados
- Conversão da variável `engine_size` para formato numérico
- Criação da variável `month_of_reference_num`, que representa os meses em formato numérico

Após essas etapas, o dataframe `dados` será utilizado como base para o treinamento dos modelos de regressão.

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# carregar base
dados = pd.read_csv('precos_carros_brasil.csv')

# remover linhas vazias
dados = dados.dropna()

# remover duplicados
dados.drop_duplicates(inplace=True)

# converter engine_size
dados['engine_size'] = dados['engine_size'].astype(str).str.replace(',', '.').astype(float)

# converter meses para número
meses = {
    'January':1,'February':2,'March':3,'April':4,
    'May':5,'June':6,'July':7,'August':8,
    'September':9,'October':10,'November':11,'December':12
}

dados['month_of_reference_num'] = dados['month_of_reference'].map(meses)

dados.shape

## 3a) Seleção das variáveis para o modelo

A variável alvo (target) será **avg_price_brl**.

As variáveis numéricas utilizadas como preditoras serão:

- year_of_reference
- engine_size
- year_model
- month_of_reference_num

As variáveis categóricas **brand**, **fuel** e **gear** serão transformadas em variáveis numéricas utilizando **One Hot Encoding**.

In [ ]:
# Importações necessárias
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
# Variável target
y = dados['avg_price_brl']

# Variáveis independentes
X = dados[['year_of_reference','engine_size','year_model','month_of_reference_num','brand','fuel','gear']]

In [ ]:
# Separação entre colunas numéricas e categóricas

numericas = ['year_of_reference','engine_size','year_model','month_of_reference_num']
categoricas = ['brand','fuel','gear']

In [ ]:
# One Hot Encoding para variáveis categóricas

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categoricas)
    ],
    remainder='passthrough'
)

## 3b) Divisão dos dados em treino e teste

Os dados foram divididos em:

- 75% para treino
- 25% para teste

In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42
)

## 3c) Treinamento dos modelos

Foram utilizados dois algoritmos de regressão:

- RandomForestRegressor
- XGBRegressor (XGBoost)

Os modelos foram treinados utilizando os dados de treino previamente separados.

In [ ]:
# Importando modelos
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

In [ ]:
# Pipeline Random Forest

pipeline_rf = Pipeline(steps=[
    ('prep', preprocessador),
    ('modelo', RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ))
])

In [ ]:
# Pipeline XGBoost

pipeline_xgb = Pipeline(steps=[
    ('prep', preprocessador),
    ('modelo', XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42
    ))
])

In [ ]:
# Treinando os modelos

pipeline_rf.fit(X_treino, y_treino)
pipeline_xgb.fit(X_treino, y_treino)

## 3d) Geração das previsões

Após o treinamento, os modelos foram utilizados para gerar previsões dos preços médios dos carros no conjunto de teste.

In [ ]:
# Predições

pred_rf = pipeline_rf.predict(X_teste)
pred_xgb = pipeline_xgb.predict(X_teste)

## 3e) Análise de importância das variáveis

A análise de importância permite identificar quais variáveis tiveram maior impacto na previsão do preço médio dos carros.

In [ ]:
import pandas as pd

# Pegando nomes das colunas após encoding

encoder = pipeline_rf.named_steps['prep'].named_transformers_['cat']
nomes_categoricos = encoder.get_feature_names_out(categoricas)

nomes_variaveis = list(nomes_categoricos) + numericas

In [ ]:
# Importância Random Forest

importancias_rf = pipeline_rf.named_steps['modelo'].feature_importances_

importancia_rf_df = pd.DataFrame({
    'variavel': nomes_variaveis,
    'importancia': importancias_rf
}).sort_values(by='importancia', ascending=False)

importancia_rf_df.head(10)

In [ ]:
# Importância XGBoost

importancias_xgb = pipeline_xgb.named_steps['modelo'].feature_importances_

importancia_xgb_df = pd.DataFrame({
    'variavel': nomes_variaveis,
    'importancia': importancias_xgb
}).sort_values(by='importancia', ascending=False)

importancia_xgb_df.head(10)

## 3f) Dê uma breve explicação (máximo de quatro linhas) sobre os resultados encontrados na análise de importância de variáveis

A análise de importância das variáveis indica quais características mais influenciam a previsão do preço médio dos veículos.

Nos dois modelos analisados, destacaram-se principalmente as variáveis `engine_size`, `year_model` e `fuel_Diesel`.

Isso indica que veículos com motores maiores tendem a possuir preços mais elevados. Além disso, o ano do modelo é um fator relevante, pois veículos mais novos geralmente possuem maior valor de mercado. O tipo de combustível também influencia o preço, sendo que veículos a diesel costumam ter valores mais altos em comparação com outros tipos de combustível.

Portanto, essas variáveis são as que mais contribuem para a capacidade preditiva dos modelos utilizados.

## 3g) Avaliação dos modelos

Para avaliar o desempenho dos modelos foram utilizadas as seguintes métricas:

- MSE (Mean Squared Error)
- MAE (Mean Absolute Error)
- R² (Coeficiente de determinação)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
# Random Forest

mse_rf = mean_squared_error(y_teste, pred_rf)
mae_rf = mean_absolute_error(y_teste, pred_rf)
r2_rf = r2_score(y_teste, pred_rf)

print("Random Forest")
print("MSE:", mse_rf)
print("MAE:", mae_rf)
print("R2:", r2_rf)

In [ ]:
# XGBoost

mse_xgb = mean_squared_error(y_teste, pred_xgb)
mae_xgb = mean_absolute_error(y_teste, pred_xgb)
r2_xgb = r2_score(y_teste, pred_xgb)

print("XGBoost")
print("MSE:", mse_xgb)
print("MAE:", mae_xgb)
print("R2:", r2_xgb)

## 3h) Dê uma breve explicação (máximo de quatro linhas) sobre qual modelo gerou o melhor resultado e a métrica de avaliação utilizada

Ao comparar os resultados obtidos, observa-se que o modelo XGBoost apresentou melhor desempenho em relação ao Random Forest.

O modelo XGBoost obteve valores menores de MSE e MAE, indicando menor erro médio nas previsões. Além disso, apresentou um valor de R² maior, mostrando maior capacidade de explicar a variação do preço médio dos veículos.

Dessa forma, conclui-se que o modelo XGBoost foi o que apresentou melhor desempenho para o problema de previsão de preços de carros.